In [1]:
import sys, os
from pathlib import Path

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if IS_KAGGLE:
    # Install packages not available on Kaggle
    %pip install -q kymatio kornia
    
    # Add repo to path (UPDATE 'deep-learning-course-project' to your dataset slug)
    repo_path = Path('/kaggle/input/deep-learning-course-project')
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
else:
    # Local: add project root to path (assumes notebook is in notebooks/)
    project_root = Path.cwd().parent
    if (project_root / 'src').exists():
        sys.path.insert(0, str(project_root))

from src.utils.config import *
from src.utils.datasets import *
from src.models.architectures.RestNet18 import *
from src.models.architectures.ScatNet18 import *
from src.utils.training import *
from src.utils.visualization import *

====================== Hyperparameters =======================
N_EPOCHS: 200
T_MAX: 200
CRITERION: CrossEntropyLoss()
DEVICE: cuda
SEED: 42
BATCH_SIZE: 128
LR: 0.001
MOMENTUM: 0.9
WEIGHT_DECAY: 0.0001
Setting seed to 42


In [2]:
DEBUG = True
SKIP_TRAINING = True
EXP_NAME = "scatresnet_by_L"
print(f"Starting experiment {EXP_NAME}. DEBUG={DEBUG}, SKIP_TRAINING={SKIP_TRAINING}")

device = DEVICE

Ls = list(range(1, 21)) # Will give a range of 1 to 63 of scattering channels out of 64

Starting experiment scatresnet_by_L. DEBUG=True, SKIP_TRAINING=True


In [3]:
test_accs : dict[int, float] = {}

# Train and save model for each data size
for L in Ls:
    # Initialize model
    scatresnet = MakeScatResNet18(L).to(device)
    MODEL_NAME = f"ScatResNet18_L{L}"

    total_params, model_size_mb = get_model_summary(scatresnet)
    print(f"Total Parameters: {total_params:,}")
    print(f"Model Size: {model_size_mb:.2f} MB")

    # Get data loaders
    trainloader, valloader, testloader, train_set, val_set, test_set = get_cifar10_loaders_and_splits()
    resnet_optimizer, resnet_scheduler = get_optimizer_and_scheduler(scatresnet)

    if not SKIP_TRAINING:
        train_model(
            model=scatresnet,
            trainloader=trainloader,
            valloader=valloader,
            optimizer=resnet_optimizer,
            scheduler=resnet_scheduler,
            device=device,
            experiment_name=EXP_NAME,
            model_name=MODEL_NAME,
            val_accuracy_storing_threshold=0,
            DEBUG=DEBUG
        )

        # Save accuracy on test set
        debug_suff = "_DEBUG" if DEBUG else ""
        load_weights(scatresnet, experiment_name=EXP_NAME, model_name=(MODEL_NAME+ debug_suff), device=device)
        test_acc = calculate_accuracy(scatresnet, testloader, device)
        test_accs[L] = test_acc
        print(f'Final test accuracy is: {test_acc:.3f}')

Total Parameters: 11,173,830
Model Size: 42.62 MB
Files already downloaded and verified
Using default 80/20% split.
Files already downloaded and verified
Original train-val size: 50000
Train size: 40000
Val size: 10000
Test size: 10000
Total Parameters: 11,173,764
Model Size: 42.62 MB
Files already downloaded and verified
Using default 80/20% split.
Files already downloaded and verified
Original train-val size: 50000
Train size: 40000
Val size: 10000
Test size: 10000
Total Parameters: 11,173,698
Model Size: 42.62 MB
Files already downloaded and verified
Using default 80/20% split.
Files already downloaded and verified
Original train-val size: 50000
Train size: 40000
Val size: 10000
Test size: 10000
Total Parameters: 11,173,632
Model Size: 42.62 MB
Files already downloaded and verified
Using default 80/20% split.
Files already downloaded and verified
Original train-val size: 50000
Train size: 40000
Val size: 10000
Test size: 10000
Total Parameters: 11,173,566
Model Size: 42.62 MB
Files 

In [4]:
# n_scat_channels = (1 + L) * 3
n_scat_channels = (1 + np.array(list(Ls))) * 3
accs = []
print(n_scat_channels)
if not SKIP_TRAINING:
    accs = list(test_accs.values())
else:
    for L in Ls:

# Plot accuracy vs L
plt.figure(figsize=(10, 5))
plt.plot(n_scat_channels/64, list(accs.values()), marker='o', linestyle='-', color='b')
plt.xlabel('%Scattering Channels')
plt.ylabel('Test Accuracy')
plt.title('ScatResNet18 Accuracy vs %Scattering Channels in First Layer')
plt.grid(True)
plt.savefig(FIGURES_PATH / f'{EXP_NAME}.png')
plt.show()

IndentationError: expected an indented block after 'for' statement on line 8 (3479075718.py, line 11)